# 14 · Bayesian inversion

Every estimate so far has been a single "best" slip model, sometimes with an
error bar attached. Bayesian inversion changes the goal: instead of one answer,
it produces the *full distribution* of models consistent with the data and our
prior beliefs. This chapter connects that idea back to the regularization we
already know, then samples a small posterior with positivity enforced, and
finishes with the diagnostics that tell you whether to trust the samples.

**Learning objectives**

By the end of the chapter, you will be able to:

- see regularized least squares as a Bayesian calculation in disguise
- explain when sampling adds something an analytic solution cannot
- draw posterior slip samples with positivity using GeoDef and BlackJAX
- read the core convergence diagnostics: R-hat, effective sample size, and
  divergences

**Prerequisites:** Chapter 08; the optional `geodef[bayes]` dependency
**Estimated time:** 90–120 minutes

> **Optional dependency.** This chapter needs JAX and BlackJAX, installed with
> `pip install "geodef[bayes]"`. The example uses a two-patch fault and short
> chains to teach the workflow, not to reach publication-quality convergence.

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Regularization was Bayesian all along

Recall the two ingredients of a Bayesian calculation. The **likelihood** says
how probable the data are for a given model; with Gaussian noise it is largest
when the model fits the data. The **prior** says how probable a model is before
seeing data; a smoothing or damping prior makes rough or large models
improbable. Bayes' theorem multiplies them into the **posterior**, the
probability of a model given the data.

Written out with Gaussian noise and a Gaussian prior, the posterior's exponent
is

$$ -\tfrac{1}{2}(G\mathbf{m}-\mathbf{d})^{\mathsf T}W(G\mathbf{m}-\mathbf{d})
   \;-\; \tfrac{1}{2}\lambda\lVert L\mathbf{m}\rVert^2, $$

which is exactly the negative of the regularized objective from Chapter 04.
Maximizing the posterior therefore gives the same slip as regularized least
squares, and the posterior's spread is the model covariance $C_m$ from
Chapter 08. In other words, we have *already been doing Bayesian inversion*; we
just reported its peak instead of its whole shape.

## 2. So why sample?

If the linear-Gaussian posterior has a known peak and known covariance, why go
to the trouble of sampling it? Because those tidy formulas break the moment the
posterior stops being a simple Gaussian bump. Three common situations do exactly
that:

- **Positivity.** Forbidding negative slip (Chapter 07) chops the Gaussian off
  at zero. Near a patch whose slip is close to zero, the posterior becomes
  lopsided, and a symmetric error bar cannot describe it.
- **Unknown hyperparameters.** If the regularization strength $\lambda$ is
  itself uncertain, we should average over it rather than fix it, which the
  analytic formula cannot do.
- **Nonlinear geometry.** When fault geometry is unknown (Chapters 09 and 10),
  the posterior can have curved ridges and multiple peaks.

**Sampling** handles all of these by drawing many models in proportion to their
posterior probability, letting the shape emerge from the draws. We will
demonstrate the first case, positivity, on a deliberately tiny problem.

## 3. A tiny two-patch problem

To keep the sampling fast and the plots readable, we use a fault with just two
patches and a small GNSS network. First the numpy-backend setup: the fault,
a known true slip, and synthetic data with 4 mm noise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef
import blackjax
import jax
import jax.numpy as jnp

geodef.backend.set_backend("numpy")
rng = np.random.default_rng(0)

In [ ]:
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=8_000.0, strike=90.0, dip=30.0,
    length=20_000.0, width=10_000.0, n_length=2, n_width=1,
)
true_slip = np.array([0.4, 1.0])     # dip slip on the two patches

In [ ]:
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.15, -117.85, 4), np.linspace(33.90, 34.10, 3)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="bayes_teaching_gnss",
)

## 4. The deterministic baseline

For comparison, first solve the ordinary way: a bounded (non-negative) damped
least-squares fit. This is the single "best" model we would normally report.

In [ ]:
linear = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="damping",
    regularization_strength=1.0, bounds=(0.0, None),
)
print(f"bounded Tikhonov slip: {np.round(linear.dip_slip, 3)}  (true {true_slip})")

## 5. Build the posterior

Now switch to the JAX backend (sampling needs it) and build a `SlipPosterior`.
This object knows the likelihood and the prior, and it enforces positivity
exactly. We use `mode="fixed"` (a single fixed $\lambda$, so the posterior is
directly comparable to the deterministic solve above) with `positive="dip"`.

In [ ]:
geodef.backend.set_backend("jax")
posterior = geodef.bayes.SlipPosterior(
    fault, gnss, components="dip", mode="fixed", positive="dip",
    regularization="damping", regularization_strength=1.0,
)
print("sampled parameters:", posterior.param_names)

The sampled parameters are not the slip values directly. Enforcing positivity
requires a reparameterization, so the sampler explores transformed variables
(here `z0`, `z1`, and `log10_sigma`); GeoDef converts samples back to physical
slip for us afterward. This is an implementation detail, but it explains why the
parameter names below are not simply "patch 1" and "patch 2."

## 6. Warm up the sampler

We use the No-U-Turn Sampler (NUTS) from BlackJAX. Like most modern samplers, it
has settings (a step size and a mass matrix) that must be tuned to the specific
posterior. **Window adaptation** does that tuning automatically during a warmup
phase, whose samples are then discarded. GeoDef offers a one-call
`geodef.bayes.sample` wrapper, but we spell the steps out here so the origin of
each diagnostic is visible.

In [ ]:
key = jax.random.PRNGKey(0)
warmup_key, chain_key = jax.random.split(key)

# Window adaptation tunes the NUTS step size and mass matrix over 200 steps.
warmup = blackjax.window_adaptation(
    blackjax.nuts, posterior.logpdf, target_acceptance_rate=0.85
)
(adapted_state, nuts_parameters), _ = warmup.run(
    warmup_key, jnp.asarray(posterior.x0), num_steps=200
)
nuts = blackjax.nuts(posterior.logpdf, **nuts_parameters)   # the tuned sampler

## 7. Run two chains

A **chain** is one sampler run: it starts somewhere and takes many steps, each
step a new model drawn near the last. Running several chains from different
starts is essential, because it is the only way to detect that the chains have
*not* mixed (Section 8). The helper below advances one chain for 150 steps,
recording each position and whether the step was a numerical failure (a
**divergence**).

In [ ]:
def run_chain(initial_position, key):
    state = nuts.init(initial_position)
    step_keys = jax.random.split(key, 150)

    # jax.lax.scan runs the 150 steps as one compiled loop for speed.
    def one_step(state, step_key):
        state, info = nuts.step(step_key, state)
        return state, (state.position, info.is_divergent)

    _, (positions, divergences) = jax.lax.scan(one_step, state, step_keys)
    return positions, divergences

In [ ]:
# Start two chains from slightly different points around the adapted state.
chain_keys = jax.random.split(chain_key, 2)
starts = adapted_state.position + 0.05 * jax.random.normal(
    chain_key, (2, posterior.n_params)
)
chain_outputs = [run_chain(starts[c], chain_keys[c]) for c in range(2)]

# Stack the chains: shape (n_chains, n_steps, n_params).
chains = np.asarray(jnp.stack([out[0] for out in chain_outputs]))
flat_samples = chains.reshape(-1, chains.shape[-1])
divergence_count = int(sum(np.asarray(out[1]).sum() for out in chain_outputs))
print(f"drew {flat_samples.shape[0]} samples across 2 chains")

## 8. Diagnose before believing

Samples are only trustworthy if the sampler actually explored the posterior.
Three diagnostics check that, and all must look right together.

- **R-hat** compares the variation *between* chains to the variation *within*
  each chain. If the chains explored the same distribution, these match and
  R-hat is near 1.0. Values much above 1 mean the chains disagree, so they have
  not converged.
- **Effective sample size (ESS)** discounts for the fact that consecutive
  samples in a chain are correlated. Hundreds of raw steps may carry the
  information of far fewer independent draws; ESS estimates how many.
- **Divergences** flag steps where the sampler's internal physics broke down.
  A handful of divergences taints the results, and the fix is usually
  reparameterization or a stronger prior, not simply keeping more samples.

In [ ]:
rhat = np.array([geodef.bayes.split_rhat(chains[:, :, k])
                 for k in range(chains.shape[-1])])
ess = np.array([geodef.bayes.effective_sample_size(chains[:, :, k])
                for k in range(chains.shape[-1])])
print("R-hat (want ~1.0):   ", np.round(rhat, 3))
print("ESS  (higher better):", np.round(ess, 0))
print("divergences (want 0):", divergence_count)

The R-hat values sit near 1.0 and there are no divergences, so for a teaching
run these short chains are acceptable. Real work would use four chains and many
more samples, and would check that the ESS is comfortably into the hundreds for
every parameter.

## 9. The posterior slip, and why it is asymmetric

Now convert the samples to physical slip and summarize each patch by its median
and a 90 percent **credible interval** (the range containing 90 percent of the
posterior probability). We overlay the truth and the deterministic estimate.

In [ ]:
slip_draws = posterior.slip_draws(flat_samples)
lower, median, upper = np.percentile(slip_draws, [5, 50, 95], axis=0)

patch = np.arange(fault.n_patches)
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
ax.errorbar(patch, median, yerr=[median - lower, upper - median],
            fmt="o", capsize=4, label="posterior median, 90% interval")
ax.plot(patch, true_slip, "ks", label="truth")
ax.plot(patch, linear.dip_slip, "x", label="bounded Tikhonov")
ax.set(xlabel="patch index", ylabel="dip slip (m)", xticks=patch,
       title="Posterior slip versus deterministic estimate")
ax.legend()
plt.show()

The credible interval is a statement of posterior probability, conditional on
this model and prior. It is *not* a frequentist confidence interval, and it is
*not* protection against a wrong fault geometry (Chapter 13 still applies). Its
key advantage over the symmetric linear error bar shows up near zero: because
positivity forbids negative slip, the interval on a small-slip patch is
lopsided, wider on the positive side, which a single standard deviation could
never represent.

## 10. A posterior predictive check

One more use of the samples: push each one back through the forward model to
predict the data, and see whether the observed data fall within the predicted
spread. This **posterior predictive check** is a basic sanity test, though
Chapter 13 warns that a flexible-enough wrong model can still pass it.

In [ ]:
prediction_draws = posterior.predict(flat_samples)
q05, q50, q95 = np.percentile(prediction_draws, [5, 50, 95], axis=0)

obs_index = np.arange(gnss.n_obs)
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
ax.fill_between(obs_index, q05, q95, alpha=0.3, label="90% predictive band")
ax.plot(obs_index, gnss.obs, ".", label="observed")
ax.plot(obs_index, q50, "-", label="predictive median")
ax.set(xlabel="observation index", ylabel="displacement (m)",
       title="Posterior predictive check")
ax.legend()
plt.show()
geodef.backend.set_backend("numpy")    # restore the default backend

The observations fall inside the predictive band, as they should for a model
that fits. Notice this checks the *data* space, not the slip; like the fit
statistics of Chapter 03, it can look healthy even when the slip is poorly
determined.

**Checkpoint.** Our two chains gave R-hat near 1.0. If instead one chain had
gotten stuck in a different region of the posterior than the other, would R-hat
be near 1.0 or much larger? Could a single chain have revealed the problem?

<details><summary>Answer</summary>

R-hat would be much larger than 1.0, because the between-chain variation (the
two chains sampling different regions) would dwarf the within-chain variation.
A single chain could not reveal this at all: with nothing to compare against, a
stuck chain looks perfectly self-consistent. That is exactly why multiple
chains from different starts are required.

</details>

## Checkpoint questions

**When is regularized least squares already a complete Bayesian answer?**

<details><summary>Answer</summary>

When the geometry is fixed, the noise is Gaussian, the prior is a proper
Gaussian, and the hyperparameters are fixed. Then the posterior is Gaussian,
its peak is the regularized solution, and its covariance is the model
covariance; nothing is gained by sampling.

</details>

**Does an R-hat near 1.0 mean the model is correct?**

<details><summary>Answer</summary>

No. R-hat diagnoses whether the *chains* have mixed, that is, whether the
sampling is trustworthy. It says nothing about whether the physics, the
likelihood, or the prior are right. A well-converged sampler can faithfully
explore a wrong model's posterior.

</details>

**Why rerun the analysis under alternative reasonable priors?**

<details><summary>Answer</summary>

To separate conclusions the data support from conclusions the prior imposed. If
a result barely changes under defensible alternative priors, the data drive it;
if it swings, the prior does, and that dependence must be reported.

</details>

## Common mistakes

- **Reporting one chain, or one seed, with no convergence evidence.** Without
  multiple chains there is no way to know the sampler mixed.
- **Calling a credible interval unconditional.** It is conditional on the fixed
  geometry and model form, exactly the assumptions Chapter 13 warns about.
- **Treating "no divergences" as proof of convergence.** No divergences is
  necessary but not sufficient; check R-hat and ESS too.
- **Showing only the preferred prior.** Hiding prior sensitivity hides how much
  of the result is assumption rather than data.

## Recap

- A fixed-geometry, Gaussian-prior inversion already has an analytic posterior;
  its peak is the regularized solution and its spread is the model covariance.
- Sampling earns its cost when positivity, unknown hyperparameters, or
  nonlinear geometry make the posterior non-Gaussian.
- Positivity makes credible intervals honestly asymmetric near zero, which a
  symmetric error bar cannot show.
- R-hat, effective sample size, and divergences must all look right before the
  samples are believed, and even then the result is conditional on the model.

This is the final chapter of the course. You now have a full toolkit: forward
modeling, linear inversion, regularization and its selection, multiple datasets
and correlated noise, constraints, uncertainty and resolution, geometry search,
alternative meshes, the interseismic problem, model-error diagnosis, and
Bayesian sampling.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/14_bayesian_inversion_solutions.ipynb`.

1. **Analytic correspondence.** Turn positivity off and compare the posterior
   mean with the (unbounded) Tikhonov solution; confirm they nearly coincide.
2. **Positivity near zero.** With positivity on and off, compare the credible
   interval on the smaller-slip patch and describe how positivity reshapes it.
3. **Prior scale.** Halve and double the prior strength and tabulate the change
   in each patch's posterior median and the total moment.
4. **Chains.** Compare one long chain with several shorter dispersed chains, and
   discuss which better reveals a convergence problem.
5. **Challenge: a deceptive predictive check.** Design a posterior predictive
   check that passes under a wrong geometry, then expose the error with a
   held-out station layout.

## Further reading

- Tarantola, A. (2005), *Inverse Problem Theory and Methods for Model Parameter
  Estimation*, on the probabilistic view of inversion.
- Hoffman, M. D., and Gelman, A. (2014), "The No-U-Turn Sampler," *Journal of
  Machine Learning Research*, 15, 1593–1623.
- Vehtari, A., et al. (2021), "Rank-normalization, folding, and localization:
  an improved R-hat," *Bayesian Analysis*, 16(2), 667–718.
- Gabry, J., et al. (2019), on visualization in Bayesian workflow.
- [GeoDef Bayesian documentation](../docs/bayes.md) for the posterior and
  sampling interfaces, including the `geodef.bayes.sample` wrapper.